In [1]:
import os

import numpy as np
import torch

In [2]:
k = "9"
d_model = 1024
device = "cuda:1"
layers = [13, 14, 15]
expansion_factor = 16
tokens_dir = "/home/fbelotti/group-sae/feature_analysis/tokens/pythia-410m/{k}"
features_dir = "/home/fbelotti/group-sae/feature_analysis/features/pythia-410m/{k}"
activations_dir = "/home/fbelotti/group-sae/feature_analysis/activations/pythia-410m/{k}"

In [3]:
torch.cuda.set_device(device)

In [4]:
# Load tokens
tokens = torch.from_numpy((np.load(os.path.join(tokens_dir.format(k="9"), "all_tokens.npy"))))

In [5]:
group = {}
for layer in layers:
    features = torch.from_numpy(
        (np.load(os.path.join(features_dir.format(k="9"), f"blocks.{layer}.hook_resid_post.npy")))
    ).reshape(-1, 128)

    activations = torch.from_numpy(
        (np.load(os.path.join(activations_dir.format(k="9"), f"blocks.{layer}.hook_resid_post.npy")))
    ).reshape(-1, 128)

    group[layer] = {
        "features": features,
        "activations": activations,
    }

In [6]:
baseline = {}
for layer in layers:
    features = torch.from_numpy(
        (np.load(os.path.join(features_dir.format(k="baseline"), f"blocks.{layer}.hook_resid_post.npy")))
    ).reshape(-1, 128)

    activations = torch.from_numpy(
        (np.load(os.path.join(activations_dir.format(k="baseline"), f"blocks.{layer}.hook_resid_post.npy")))
    ).reshape(-1, 128)

    baseline[layer] = {
        "features": features,
        "activations": activations,
    }

In [84]:
from transformers import AutoTokenizer

# Load the tokenizer for pythia-410m
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-410m")


def get_top_activating_tokens(
    features: torch.Tensor,
    activations: torch.Tensor,
    feature_idx,
    top_k=5,
    context_length=16,
    print_highlighted_only: bool = True,
):
    """
    Find the top-k most activating tokens for a given feature and decode them with context.

    Args:
        feature_idx: The feature index to analyze
        top_k: Number of top activating tokens to return
        context_length: Number of tokens before and after to show as context
    """
    print(f"=== TOP {top_k} ACTIVATING TOKENS FOR FEATURE {feature_idx} ===")

    # Find all positions where this feature appears
    feature_positions = torch.where(features == feature_idx)

    if len(feature_positions[0]) == 0:
        print(f"Feature {feature_idx} never activates!")
        return

    # Get the activation values for this feature at all positions where it appears
    activation_values = activations[feature_positions]

    # Get top-k positions with highest activations
    top_k_indices = torch.topk(activation_values, min(top_k, len(activation_values))).indices

    print(f"Feature {feature_idx} activates {len(feature_positions[0])} times")
    print(f"Activation range: {activation_values.min():.4f} to {activation_values.max():.4f}")
    print()

    for rank, idx in enumerate(top_k_indices):
        token_idx = feature_positions[0][idx].item()
        activation_value = activation_values[idx].item()

        # Calculate the original token position in the sequence
        # Since we reshaped to (-1, 128), we need to find the original sequence position
        batch_size = tokens.shape[1] if len(tokens.shape) > 1 else len(tokens)
        original_token_pos = token_idx % batch_size
        sequence_idx = token_idx // batch_size

        # Get the context around this token
        start_pos = max(0, original_token_pos - context_length)
        end_pos = min(
            len(tokens) if len(tokens.shape) == 1 else tokens.shape[1],
            original_token_pos + context_length + 1,
        )

        if len(tokens.shape) == 1:
            # 1D tokens array
            context_tokens = tokens[start_pos:end_pos]
            target_token = tokens[original_token_pos]
        else:
            # 2D tokens array
            context_tokens = tokens[sequence_idx, start_pos:end_pos]
            target_token = tokens[sequence_idx, original_token_pos]

        # Decode the tokens
        context_text = tokenizer.decode(context_tokens.cpu().numpy(), skip_special_tokens=False)
        target_text = tokenizer.decode([target_token.cpu().item()], skip_special_tokens=False)

        if not print_highlighted_only:
            print(f"Rank {rank + 1}: Activation = {activation_value:.4f}")
            print(
                f"  Token position: {original_token_pos} (sequence {sequence_idx if len(tokens.shape) > 1 else 0})"
            )
            print(f"  Target token: '{target_text}' (ID: {target_token.item()})")
            print(f"  Context: {repr(context_text)}")

        # Highlight the target token in context
        target_pos_in_context = original_token_pos - start_pos
        context_tokens_list = context_tokens.cpu().numpy().tolist()
        decoded_context_tokens = [
            tokenizer.decode([t], skip_special_tokens=False) for t in context_tokens_list
        ]

        highlighted_context = ""
        for i, token_text in enumerate(decoded_context_tokens):
            if i == target_pos_in_context:
                highlighted_context += f">>>{token_text}<<<"
            else:
                highlighted_context += token_text

        print(f"  Highlighted: {repr(highlighted_context)}")
        print()

In [42]:
def select_features_by_activation_percent(features, min_percent, max_percent, return_stats=False):
    """
    Select features with activation percentage between min_percent and max_percent.

    Args:
        min_percent: Minimum activation percentage (0-100)
        max_percent: Maximum activation percentage (0-100)
        return_stats: If True, return additional statistics

    Returns:
        If return_stats=False: array of feature indices
        If return_stats=True: tuple of (feature_indices, activation_counts, activation_percentages)
    """
    # Compute feature distribution
    bs = 256
    features_dist = torch.zeros(d_model * expansion_factor, device="cuda")
    for i in range(0, features.shape[0], bs):
        features_dist.scatter_add_(
            0,
            features[i : i + bs].to("cuda").view(-1).long(),
            torch.ones(features[i : i + bs].numel(), device="cuda"),
        )

    # Get activation counts for all features
    features_dist_cpu = features_dist.cpu().numpy()

    # Calculate activation percentages
    activation_percentages = (features_dist_cpu / features_dist.shape[0]) * 100

    # Select features within the specified range
    mask = (activation_percentages >= min_percent) & (activation_percentages <= max_percent)
    selected_indices = np.where(mask)[0]

    return features_dist, selected_indices

In [53]:
dist, indices = select_features_by_activation_percent(group[14]["features"], 0.1, 0.15)

In [54]:
indices

array([  101,   154,   184,   550,   792,  1027,  1181,  1234,  1626,
        1722,  1737,  1779,  1851,  1995,  2007,  2093,  2181,  2282,
        2550,  2730,  2986,  3007,  3192,  3269,  3293,  3446,  3525,
        3529,  3660,  3819,  4023,  4118,  4686,  4706,  4836,  4911,
        5195,  5226,  5228,  5330,  5563,  5666,  5669,  5694,  5863,
        5910,  6004,  6082,  6085,  6124,  6657,  7010,  7060,  7067,
        7212,  7233,  7568,  7757,  7870,  7902,  8009,  8266,  8340,
        8401,  8582,  8646,  8667,  8697,  8860,  8878,  9010,  9094,
        9170,  9238,  9288,  9340,  9623,  9671, 10103, 10441, 10569,
       10605, 10739, 10891, 11000, 11040, 11479, 11573, 11656, 11687,
       11764, 11954, 11980, 12004, 12322, 12345, 12509, 12578, 12752,
       12893, 12899, 13014, 13192, 13365, 13406, 13504, 13514, 13705,
       13723, 13759, 13867, 13903, 13940, 13944, 14834, 14856, 14975,
       14982, 15106, 15513, 15696, 15781, 15949, 16034, 16071, 16174,
       16207, 16334]

In [55]:
"""Feature 225 is highly selective for the period token '.' (ID: 15) when
it appears immediately before function calls like Fatalf, Error, or Errorf in Go-like error handling code.
This feature likely detects the syntactic pattern of method or function invocation following a period, especially in error reporting statements.
"""

"""Feature 1024 is highly selective for the token ' 1' (ID: 337), especially when it appears in mathematical or sorting contexts.
The top activations occur when ' 1' is part of lists or sequences of numbers, often in instructions to sort or order numbers.
This feature likely detects the presence of the number 1 within numeric lists or sorting tasks.
"""

"""Feature 0 detects numeric values (especially 2-4 digit numbers) in scientific and technical contexts.
The top activations occur on tokens like ' 100', ' 170', ' 95', '4000', and ' 40' when they appear
with units of measurement (nM, °C, K, ms, GeV) or in scientific notation contexts.
This feature likely captures the pattern of numerical quantities with associated units in academic/scientific text."""

get_top_activating_tokens(
    group[14]["features"], group[14]["activations"], 101, top_k=5, context_length=16
)

=== TOP 5 ACTIVATING TOKENS FOR FEATURE 101 ===
Feature 101 activates 22 times
Activation range: 0.0198 to 0.2747

Rank 1: Activation = 0.2747
  Token position: 170230 (sequence 0)
  Target token: ' Perm' (ID: 22689)
  Context: ' license is not in English, it is recommended that you get an International Driving Permit (IDP).\n\n### Bicycle\n\nABike lanes'
  Highlighted: ' license is not in English, it is recommended that you get an International Driving>>> Perm<<<it (IDP).\n\n### Bicycle\n\nABike lanes'

Rank 2: Activation = 0.2654
  Token position: 234135 (sequence 0)
  Target token: ' perm' (ID: 8143)
  Context: ' from a global knowledge base that can instantaneously transmit new local food production ideas, permaculture strategies and farming “best practices” to anyone, anywhere on the globe'
  Highlighted: ' from a global knowledge base that can instantaneously transmit new local food production ideas,>>> perm<<<aculture strategies and farming “best practices” to anyone, anywhere o

### Concordance

In [ ]:
def compute_jaccard_similarity_cuda(
    baseline_features,
    group_features,
    M,
    batch_size=2048,
    similarity_threshold=0.5,
    min_activation_count=1,
):
    """
    Compute Jaccard similarity matrix and identify baseline-only and group-only features using efficient set operations.

    Args:
        baseline_features: Baseline-SAE features tensor (N, K) on CPU
        group_features: Group-SAE features tensor (N, K) on CPU
        M: Number of features (d_model * expansion_factor)
        batch_size: Batch size for memory efficiency
        similarity_threshold: Minimum Jaccard similarity to consider features as matched
        min_activation_count: Minimum activations to consider a feature as active

    Returns:
        dict with jaccard_similarity, baseline_only_features, group_only_features, shared_features
    """
    # Move to device and ensure int type
    baseline_act = baseline_features.to(device).int()
    group_act = group_features.to(device).int()

    N_tokens = baseline_act.shape[0]
    print(f"Computing Jaccard similarity matrix for {N_tokens:,} tokens...")
    print(f"Feature space size: {M} x {M} = {M*M:,} entries")

    # First, compute feature counts to identify active features
    print("Computing feature counts...")
    count_baseline = torch.bincount(baseline_act.reshape(-1), minlength=M)
    count_group = torch.bincount(group_act.reshape(-1), minlength=M)

    # Convert to CPU for set operations
    count_baseline_cpu = count_baseline.cpu().numpy()
    count_group_cpu = count_group.cpu().numpy()

    # Create sets of active features using efficient numpy operations
    baseline_active_set = set(np.where(count_baseline_cpu >= min_activation_count)[0])
    group_active_set = set(np.where(count_group_cpu >= min_activation_count)[0])

    print(f"Active baseline features: {len(baseline_active_set)}")
    print(f"Active group features: {len(group_active_set)}")

    # Use set operations to find feature relationships
    # Features that exist in both SAEs (but may have different similarities)
    potentially_shared = baseline_active_set & group_active_set

    # Features that only exist in one SAE or the other
    baseline_only_candidates = baseline_active_set - group_active_set
    group_only_candidates = group_active_set - baseline_active_set

    print(f"Features present in both SAEs: {len(potentially_shared)}")
    print(f"Baseline-only candidates: {len(baseline_only_candidates)}")
    print(f"Group-only candidates: {len(group_only_candidates)}")

    # Allocate global histogram vector for the AND matrix (flattened)
    global_hist = torch.zeros(M * M, device=device, dtype=torch.int32)

    # Process tokens in batches to avoid huge memory allocations
    print(f"Processing in batches of {batch_size}...")
    for i in range(0, N_tokens, batch_size):
        if i % (batch_size * 10) == 0:
            print(f"  Batch {i//batch_size + 1}/{(N_tokens + batch_size - 1)//batch_size}")

        batch_end = min(i + batch_size, N_tokens)

        # Get a batch of tokens, shape: (B, K)
        baseline_batch = baseline_act[i:batch_end]
        group_batch = group_act[i:batch_end]

        # Compute the outer (Cartesian) product for each token in the batch
        batch_linear_idx = baseline_batch.unsqueeze(2) * M + group_batch.unsqueeze(1)
        batch_linear_idx = batch_linear_idx.reshape(-1)

        # Count co-occurrences in this batch
        batch_hist = torch.bincount(batch_linear_idx, minlength=M * M)
        global_hist += batch_hist.to(torch.int32)

        # Clean up batch memory
        del baseline_batch, group_batch, batch_linear_idx, batch_hist
        if i % (batch_size * 4) == 0:
            torch.cuda.empty_cache()

    # Reshape global histogram into the AND matrix of shape (M, M)
    and_matrix = global_hist.reshape(M, M).to(torch.int32)
    del global_hist
    torch.cuda.empty_cache()

    # OR matrix: OR(i,j) = count_baseline[i] + count_group[j] - and_matrix[i,j]
    or_matrix = count_baseline.view(M, 1) + count_group.view(1, M) - and_matrix

    # Compute Jaccard similarity matrix
    jaccard_similarity = and_matrix.float() / (or_matrix.float() + 1e-8)
    jaccard_cpu = jaccard_similarity.cpu().numpy()

    # Now refine the categorization based on actual Jaccard similarities
    print("Analyzing feature similarities...")

    # Convert active sets to sorted lists for indexing
    baseline_active_list = sorted(list(baseline_active_set))
    group_active_list = sorted(list(group_active_set))

    shared_features = []
    baseline_matched = set()
    group_matched = set()

    # For each baseline feature, find best group match
    for baseline_feat in baseline_active_list:
        if len(group_active_list) == 0:
            continue

        # Get similarities only for active group features
        similarities = jaccard_cpu[baseline_feat, group_active_list]
        best_idx = similarities.argmax()
        best_similarity = similarities[best_idx]
        best_group_feat = group_active_list[best_idx]

        if best_similarity >= similarity_threshold:
            shared_features.append((baseline_feat, best_group_feat, best_similarity))
            baseline_matched.add(baseline_feat)
            group_matched.add(best_group_feat)

    # Final categorization using set operations
    baseline_only = baseline_active_set - baseline_matched
    group_only = group_active_set - group_matched

    # Convert back to sorted lists
    baseline_only = sorted(list(baseline_only))
    group_only = sorted(list(group_only))

    print("Final results:")
    print(f"Shared features (high similarity): {len(shared_features)}")
    print(f"Baseline-only features: {len(baseline_only)}")
    print(f"Group-only features: {len(group_only)}")

    # Additional statistics
    total_active_baseline = len(baseline_active_set)
    total_active_group = len(group_active_set)

    print("\nCoverage statistics:")
    print(
        f"Baseline coverage: {len(baseline_matched)}/{total_active_baseline} "
        f"({100*len(baseline_matched)/total_active_baseline:.1f}%)"
    )
    print(
        f"Group coverage: {len(group_matched)}/{total_active_group} "
        f"({100*len(group_matched)/total_active_group:.1f}%)"
    )

    # Clean up
    del baseline_act, group_act, and_matrix, or_matrix, count_baseline, count_group
    torch.cuda.empty_cache()

    return {
        "jaccard_similarity": jaccard_similarity,
        "shared_features": shared_features,
        "baseline_only_features": baseline_only,
        "group_only_features": group_only,
        "baseline_feature_counts": count_baseline_cpu,
        "group_feature_counts": count_group_cpu,
        "baseline_active_set": baseline_active_set,
        "group_active_set": group_active_set,
        "potentially_shared": potentially_shared,
        "stats": {
            "total_baseline_active": total_active_baseline,
            "total_group_active": total_active_group,
            "shared_count": len(shared_features),
            "baseline_only_count": len(baseline_only),
            "group_only_count": len(group_only),
            "potentially_shared_count": len(potentially_shared),
            "baseline_coverage": (
                len(baseline_matched) / total_active_baseline if total_active_baseline > 0 else 0
            ),
            "group_coverage": (
                len(group_matched) / total_active_group if total_active_group > 0 else 0
            ),
        },
    }


def compare_group_vs_baseline_jaccard(
    group_layer,
    baseline_layers=["13", "14", "15"],
    similarity_threshold=0.8,
    min_activation_count=1,
    batch_size=4096,
):
    """
    Compare Group-SAE features with Baseline-SAE features using efficient CUDA Jaccard similarity.

    Args:
        group_layer: Which layer of Group-SAE to analyze
        baseline_layers: List of baseline layers to compare against
        similarity_threshold: Minimum Jaccard similarity (recommend 0.1-0.5)
        min_activation_count: Minimum activations to consider a feature as active
    """
    from collections import defaultdict

    results = {
        "shared_features": defaultdict(list),
        "baseline_only": defaultdict(list),
        "group_layer": group_layer,
        "concordance": {},
    }

    M = d_model * expansion_factor

    # Load Group-SAE features
    print(f"Loading Group-SAE features for layer {group_layer}...")
    try:
        group_features_path = f"/home/fbelotti/group-sae/feature_analysis/features/pythia-410m/{k}/blocks.{group_layer}.hook_resid_post.npy"
        group_features = torch.from_numpy(np.load(group_features_path))
        group_features = group_features.reshape(-1, 128)
        print(f"Group-SAE layer {group_layer} features shape: {group_features.shape}")
    except FileNotFoundError:
        print(f"Error: Group-SAE features for layer {group_layer} not found!")
        return results

    # Get Group-SAE active features for filtering
    print(f"Computing Group-SAE layer {group_layer} feature counts...")
    group_dist = torch.zeros(M, device=device)
    cuda_batch_size = 1024

    for i in range(0, group_features.shape[0], cuda_batch_size):
        batch_end = min(i + cuda_batch_size, group_features.shape[0])
        batch_features = group_features[i:batch_end].to(device)
        batch_dist = torch.bincount(batch_features.view(-1), minlength=M)
        group_dist += batch_dist
        del batch_features, batch_dist
        if i % (cuda_batch_size * 4) == 0:
            torch.cuda.empty_cache()

    group_dist_cpu = group_dist.cpu().numpy()
    group_active_indices = np.where(group_dist_cpu >= min_activation_count)[0]
    print(f"Group-SAE layer {group_layer} active features: {len(group_active_indices)}")
    del group_dist
    torch.cuda.empty_cache()

    # Process each baseline layer
    for baseline_layer in baseline_layers:
        print(f"\n=== Comparing with Baseline layer {baseline_layer} ===")

        try:
            baseline_path = f"/home/fbelotti/group-sae/feature_analysis/features/pythia-410m/baseline/blocks.{baseline_layer}.hook_resid_post.npy"
            baseline_features = torch.from_numpy(np.load(baseline_path))
            baseline_features = baseline_features.reshape(-1, 128)
            print(f"Baseline layer {baseline_layer} features shape: {baseline_features.shape}")
        except FileNotFoundError:
            print(f"Warning: Baseline features for layer {baseline_layer} not found, skipping...")
            continue

        # Get baseline active features
        print(f"Computing Baseline layer {baseline_layer} feature counts...")
        baseline_dist = torch.zeros(M, device=device)
        for i in range(0, baseline_features.shape[0], cuda_batch_size):
            batch_end = min(i + cuda_batch_size, baseline_features.shape[0])
            batch_features = baseline_features[i:batch_end].to(device)
            batch_dist = torch.bincount(batch_features.view(-1), minlength=M)
            baseline_dist += batch_dist
            del batch_features, batch_dist
            if i % (cuda_batch_size * 4) == 0:
                torch.cuda.empty_cache()

        baseline_dist_cpu = baseline_dist.cpu().numpy()
        baseline_active_indices = np.where(baseline_dist_cpu >= min_activation_count)[0]
        print(f"Baseline layer {baseline_layer} active features: {len(baseline_active_indices)}")
        del baseline_dist
        torch.cuda.empty_cache()

        # Compute full Jaccard similarity matrix using efficient CUDA method
        jaccard_stats = compute_jaccard_similarity_cuda(
            baseline_features,
            group_features,
            M,
            batch_size=batch_size,
            similarity_threshold=similarity_threshold,
            min_activation_count=min_activation_count,
        )
        jaccard_matrix = jaccard_stats["jaccard_similarity"]

        # Move to CPU for analysis
        del jaccard_matrix, jaccard_stats["jaccard_similarity"]
        torch.cuda.empty_cache()

        results["concordance"][baseline_layer] = jaccard_stats
    torch.cuda.empty_cache()
    return results

In [95]:
# Run with optimized parameters for 16GB VRAM
jaccard_results_13 = compare_group_vs_baseline_jaccard(
    group_layer=13,
    baseline_layers=["13", "14", "15"],
    similarity_threshold=0.4,
    min_activation_count=1,
    batch_size=4096,
)

Loading Group-SAE features for layer 13...
Group-SAE layer 13 features shape: torch.Size([1003520, 128])
Computing Group-SAE layer 13 feature counts...
Group-SAE layer 13 active features: 16023

=== Comparing with Baseline layer 13 ===
Baseline layer 13 features shape: torch.Size([1003520, 128])
Computing Baseline layer 13 feature counts...
Baseline layer 13 active features: 16239
Computing Jaccard similarity matrix for 1,003,520 tokens...
Feature space size: 16384 x 16384 = 268,435,456 entries
Computing feature counts...
Active baseline features: 16239
Active group features: 16023
Features present in both SAEs: 15883
Baseline-only candidates: 356
Group-only candidates: 140
Processing in batches of 4096...
  Batch 1/245
  Batch 11/245
  Batch 21/245
  Batch 31/245
  Batch 41/245
  Batch 51/245
  Batch 61/245
  Batch 71/245
  Batch 81/245
  Batch 91/245
  Batch 101/245
  Batch 111/245
  Batch 121/245
  Batch 131/245
  Batch 141/245
  Batch 151/245
  Batch 161/245
  Batch 171/245
  Batch

In [94]:
# Run with optimized parameters for 16GB VRAM
jaccard_results_14 = compare_group_vs_baseline_jaccard(
    group_layer=14,
    baseline_layers=["13", "14", "15"],
    similarity_threshold=0.4,
    min_activation_count=1,
    batch_size=4096,
)

Loading Group-SAE features for layer 14...
Group-SAE layer 14 features shape: torch.Size([1003520, 128])
Computing Group-SAE layer 14 feature counts...
Group-SAE layer 14 active features: 16197

=== Comparing with Baseline layer 13 ===
Baseline layer 13 features shape: torch.Size([1003520, 128])
Computing Baseline layer 13 feature counts...
Baseline layer 13 active features: 16239
Computing Jaccard similarity matrix for 1,003,520 tokens...
Feature space size: 16384 x 16384 = 268,435,456 entries
Computing feature counts...
Active baseline features: 16239
Active group features: 16197
Features present in both SAEs: 16055
Baseline-only candidates: 184
Group-only candidates: 142
Processing in batches of 4096...
  Batch 1/245
  Batch 11/245
  Batch 21/245
  Batch 31/245
  Batch 41/245
  Batch 51/245
  Batch 61/245
  Batch 71/245
  Batch 81/245
  Batch 91/245
  Batch 101/245
  Batch 111/245
  Batch 121/245
  Batch 131/245
  Batch 141/245
  Batch 151/245
  Batch 161/245
  Batch 171/245
  Batch

In [96]:
jaccard_results_13["concordance"]["13"]["shared_features"].sort(key=lambda x: x[2], reverse=True)
jaccard_results_13["concordance"]["14"]["shared_features"].sort(key=lambda x: x[2], reverse=True)
jaccard_results_13["concordance"]["15"]["shared_features"].sort(key=lambda x: x[2], reverse=True)

In [97]:
jaccard_results_14["concordance"]["13"]["shared_features"].sort(key=lambda x: x[2], reverse=True)
jaccard_results_14["concordance"]["14"]["shared_features"].sort(key=lambda x: x[2], reverse=True)
jaccard_results_14["concordance"]["15"]["shared_features"].sort(key=lambda x: x[2], reverse=True)

In [98]:
jaccard_results_14["concordance"]["15"]["shared_features"][-1000:-900]

[(3414, 12019, 0.46886447),
 (12076, 12413, 0.46882525),
 (8615, 10833, 0.4688174),
 (8513, 13865, 0.4688172),
 (3156, 12257, 0.46872023),
 (16078, 9927, 0.46866363),
 (4083, 7029, 0.46857774),
 (12993, 17, 0.46853945),
 (7448, 15353, 0.46838886),
 (15672, 1601, 0.46834573),
 (11672, 948, 0.4683281),
 (2014, 9682, 0.46819392),
 (2027, 6959, 0.46816975),
 (16054, 9521, 0.46801475),
 (15071, 1983, 0.46801096),
 (11558, 14741, 0.46786708),
 (5990, 15508, 0.467799),
 (16217, 9271, 0.46774194),
 (16074, 2210, 0.4676342),
 (2840, 9306, 0.46760345),
 (2689, 9938, 0.46747968),
 (12361, 13954, 0.46746987),
 (11023, 15906, 0.4667046),
 (3183, 3452, 0.46663535),
 (8505, 11174, 0.46652806),
 (462, 4855, 0.46651182),
 (1495, 8387, 0.46632123),
 (10208, 10910, 0.46630126),
 (3221, 8647, 0.4661805),
 (6289, 6164, 0.46606913),
 (5449, 13309, 0.46588877),
 (1993, 8525, 0.46568626),
 (10748, 1617, 0.46563658),
 (6099, 6563, 0.46559232),
 (7167, 3527, 0.4654579),
 (3897, 10495, 0.4651993),
 (2407, 14873,

In [99]:
get_top_activating_tokens(
    baseline[15]["features"], baseline[15]["activations"], 3156, top_k=5, context_length=16
)

=== TOP 5 ACTIVATING TOKENS FOR FEATURE 3156 ===
Feature 3156 activates 2654 times
Activation range: 0.0091 to 0.3777

  Highlighted: ' Florida" is a bit of a misnomer; she was already there.>>> Most<<< hurricane cloud shields are at least 300 miles in diameter, but it\'s only the'

  Highlighted: ' are affiliated with paying millions to candidates, who are just running for legacy purposes.>>> Most<<< candidates/politicians, are Masters of Talk. Getting very little done, to'

  Highlighted: ' grew up in a loving and close family with his parents and his baby sister.>>> Most<<< of his extended family, aunts uncles and grandparents, live in the same'

  Highlighted: ' had bigger ideas." "She\'s always wanted to see the world." "And>>> most<<< of all, she\'s wanted to live on a shore, on a coast "'

  Highlighted: '\n\n----"I got a lot of really good ideas, problem is,>>> most<<< of them suck."\n- George Carlin\n\n2. Gojira'



In [100]:
get_top_activating_tokens(
    group[14]["features"], group[14]["activations"], 12257, top_k=5, context_length=16
)

=== TOP 5 ACTIVATING TOKENS FOR FEATURE 12257 ===
Feature 12257 activates 1971 times
Activation range: 0.0119 to 0.3970

  Highlighted: ' had bigger ideas." "She\'s always wanted to see the world." "And>>> most<<< of all, she\'s wanted to live on a shore, on a coast "'

  Highlighted: ' to, I wanted to see them, I wanted to feel them, and\n>>>most<<< of all I wanted to taste them.\nShe pulled me back into the kiss'

  Highlighted: ' are affiliated with paying millions to candidates, who are just running for legacy purposes.>>> Most<<< candidates/politicians, are Masters of Talk. Getting very little done, to'

  Highlighted: ' Florida" is a bit of a misnomer; she was already there.>>> Most<<< hurricane cloud shields are at least 300 miles in diameter, but it\'s only the'

  Highlighted: ' God as a loving therapist Who is always there to listen, to understand, and>>> most<<< importantly, not to judge us. This verse reminds us that above all, the'

